# Spark DataFrame API


Освоить цепочки преобразований, типы, пропуски и группировку через API DataFrame.

Импортируйте функции Spark в каждой ячейке, которая запускает отдельный контекст.

Ограничения: Когда нужна выразительная типизированная цепочка Python или общие функции с другими операциями Spark. Не рассчитывайте, что промежуточное значение вычислено сразу; действие запускается только `show`, `count`, `first` или похожим вызовом.


## Основные понятия
DataFrame описывает новый набор данных после каждого вызова. `select`, `filter`, `withColumn` и `groupBy` соответствуют выбору, отбору, вычислению и группировке.


## Примеры


**1. Цепочка преобразований**

Оставьте нужные поля, вычислите минуты и общий счёт.

Сопоставьте поля, значения и размер выдачи с ожидаемым поведением операции; при расхождении сначала проверьте схему.


In [ ]:
from lakehouse_spark import SparkSessionContext
from pyspark.sql import functions as F


with SparkSessionContext('example-df-chain') as spark:
    result = (spark.table("lakehouse.raw.pg__matches")
        .select("match_id", "duration", "radiant_score", "dire_score", "radiant_win")
        .filter(F.col("duration").between(600, 7200))
        .withColumn("duration_minutes", F.round(F.col("duration") / 60.0, 1))
        .withColumn("total_kills", F.col("radiant_score") + F.col("dire_score"))
        .orderBy(F.desc("total_kills")))
    result.show(10, truncate=False)


**2. Типы и пропуски**

Явно приведите ключи и замените пропуски в числовых показателях.

Сопоставьте поля, значения и размер выдачи с ожидаемым поведением операции; при расхождении сначала проверьте схему.


In [ ]:
from lakehouse_spark import SparkSessionContext
from pyspark.sql import functions as F


with SparkSessionContext('example-df-types') as spark:
    frame = (spark.table("lakehouse.raw.pg__match_players")
        .select(F.col("match_id").cast("long"), F.col("hero_id").cast("int"),
                F.coalesce("kills", F.lit(0)).alias("kills"), F.coalesce("deaths", F.lit(0)).alias("deaths")))
    frame.printSchema()
    frame.limit(10).show(truncate=False)


**3. Несколько показателей**

Соберите число игр и средние значения одним вызовом группировки.

Сопоставьте поля, значения и размер выдачи с ожидаемым поведением операции; при расхождении сначала проверьте схему.


In [ ]:
from lakehouse_spark import SparkSessionContext
from pyspark.sql import functions as F


with SparkSessionContext('example-df-aggregation') as spark:
    (spark.table("lakehouse.raw.pg__match_players")
        .filter(F.col("hero_id").isNotNull())
        .groupBy("hero_id")
        .agg(F.count("*").alias("games"), F.round(F.avg("kills"), 2).alias("average_kills"),
             F.round(F.avg("gold_per_min"), 2).alias("average_gold"))
        .orderBy(F.desc("games"))
        .show(15, truncate=False))


**4. Условия и категории**

Преобразуйте числовое поле в понятные категории через `when`.

Сопоставьте поля, значения и размер выдачи с ожидаемым поведением операции; при расхождении сначала проверьте схему.


In [ ]:
from lakehouse_spark import SparkSessionContext
from pyspark.sql import functions as F


with SparkSessionContext('example-df-categories') as spark:
    result = (spark.table("lakehouse.raw.pg__matches")
        .select("match_id", "duration")
        .withColumn("duration_group", F.when(F.col("duration") < 1800, "short")
                    .when(F.col("duration") < 3600, "medium").otherwise("long"))
        .groupBy("duration_group").count().orderBy("duration_group"))
    result.show(truncate=False)


**5. Окончательная схема**

Проверьте имена и типы результата перед передачей его другой операции.

Сопоставьте поля, значения и размер выдачи с ожидаемым поведением операции; при расхождении сначала проверьте схему.


In [ ]:
from lakehouse_spark import SparkSessionContext
from pyspark.sql import functions as F


with SparkSessionContext('example-df-schema') as spark:
    spark.table("lakehouse.raw.pg__matches").select("match_id", "duration", "radiant_win").printSchema()
